# Final Project: PySpark MLlib Modeling and Streaming Predictions
## Jacob A. Fericy

In [ ]:
from pathlib import Path
import shutil
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql.functions import col

from pyspark.ml import Pipeline
from pyspark.ml.feature import SQLTransformer, Binarizer, StringIndexer, OneHotEncoder, VectorAssembler, PCA
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

In [ ]:
spark = SparkSession.builder.appName("FinalProjectFericy").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

BASE_DIR = Path.cwd()
ML_FILE = BASE_DIR / "power_ml_data.csv"
STREAMING_FILE = BASE_DIR / "power_streaming_data.csv"
STREAM_INPUT_DIR = BASE_DIR / "power_stream_input"

print("Current folder:", BASE_DIR)
print("Modeling file exists:", ML_FILE.exists())
print("Streaming source file exists:", STREAMING_FILE.exists())

## 2. Read the modeling data with pandas

The assignment asks for `pd.read_csv()` first, then conversion to a Spark data frame.

In [ ]:
power_pd = pd.read_csv(ML_FILE)
print(power_pd.shape)
power_pd.head()

In [ ]:
power_pd.dtypes

In [ ]:
## 3. Convert the pandas data frame to Spark

In [ ]:
power_sdf = spark.createDataFrame(power_pd)
power_sdf.printSchema()
power_sdf.show(5)

In [ ]:
pca_input_cols = ["Temperature", "Humidity", "Wind_Speed", "General_Diffuse_Flows", "Diffuse_Flows"]

sql_transformer = SQLTransformer(statement="""
    SELECT
        CAST(Hour AS DOUBLE) AS Hour_Double,
        CAST(Month AS DOUBLE) AS Month_Double,
        Temperature,
        Humidity,
        Wind_Speed,
        General_Diffuse_Flows,
        Diffuse_Flows,
        Power_Zone_1,
        Power_Zone_2,
        Power_Zone_3 AS label
    FROM __THIS__
""")

hour_binarizer = Binarizer(threshold = 6.5, inputCol = "Hour_Double", outputCol = "Hour_Binary")
month_indexer = StringIndexer(inputCol = "Month_Double", outputCol = "Month_Index", handleInvalid = "keep")
month_encoder = OneHotEncoder(inputCols = ["Month_Index"], outputCols = ["Month_OHE"], handleInvalid = "keep")
pca_assembler = VectorAssembler(inputCols = pca_input_cols, outputCol = "pca_input_features", handleInvalid = "keep")
pca = PCA(k = 2, inputCol = "pca_input_features", outputCol = "pca_features")

feature_assembler = VectorAssembler(
    inputCols = ["pca_features", "Hour_Binary", "Power_Zone_1", "Power_Zone_2", "Month_OHE"],
    outputCol = "features",
    handleInvalid = "keep"
)

lr = LinearRegression(featuresCol = "features", labelCol = "label", predictionCol = "prediction")

pipeline = Pipeline(stages=[
    sql_transformer,
    hour_binarizer,
    month_indexer,
    month_encoder,
    pca_assembler,
    pca,
    feature_assembler,
    lr
])